🎬 🎓 SCENARIO: “College Smart Assistant System”
🏫 Background Story

A college builds an AI-powered student assistant.

👉 Students can ask:

“What is my attendance?”
“What are my marks?”

👉 Instead of manually checking portals,
👉 AI fetches it instantly.

In [1]:

# ================================
# STEP 1: Install Gradio
# ================================
# !pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def mcp_agent(message, student_id, history):

    # Simulate logged-in user (for demo)
    user_id = student_id

    # Security Check
    if not secure_access(user_id, student_id):
        return "🚫 Access Denied", history

    # Tool Invocation Logic
    message_lower = message.lower()

    if "attendance" in message_lower:
        response = get_attendance(student_id)

    elif "marks" in message_lower:
        response = get_marks(student_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about attendance or marks."

    else:
        response = "🤖 I can help with attendance or marks."

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 Student MCP Agent (Gradio Version)")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch App (Public URL)
# ================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# STUDENT MCP USING GROQ INTEGRATION

In [10]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from dotenv import load_dotenv
import os
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = "⚠️ Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
def mcp_agent(message, student_id, history):

    if not student_id:
        response = "⚠️ Please enter Student ID"
        history = history + [
            {"role": "assistant", "content": response}
        ]
        return "", history, history

    tool = decide_tool(message)

    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = "⚠️ LLM failed"

    else:
        response = "🤖 I can help with attendance or marks."


    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history, history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 MCP Agent with Groq")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot, state]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “Hospital Smart Assistant System”
🏥 Background Story
A large hospital deploys an AI-powered patient assistant.
👉 Patients can ask:
- “What is my appointment schedule?”
- “What are my latest test results?”
👉 Instead of calling reception or logging into multiple portals,
👉 AI fetches it instantly, providing secure, real-time updates.

In [11]:
# ================================
# STEP 1: Install Gradio
# ================================
# !pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
patients = {
    "P101": {
        "name": "Rahul",
        "appointment": "10 Oct, 10:30 AM - Dr. Sharma",
        "test_results": "Blood Test: Normal"
    },
    "P102": {
        "name": "Priya",
        "appointment": "12 Oct, 2:00 PM - Dr. Mehta",
        "test_results": "X-Ray: Minor fracture"
    }
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_appointment(patient_id):
    if patient_id in patients:
        return f"📅 Appointment: {patients[patient_id]['appointment']}"
    return "❌ Patient not found"


def get_test_results(patient_id):
    if patient_id in patients:
        return f"🧪 Test Results: {patients[patient_id]['test_results']}"
    return "❌ Patient not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def hospital_agent(message, patient_id, history):

    # Simulate logged-in user
    user_id = patient_id

    # Security Check
    if not secure_access(user_id, patient_id):
        return "🚫 Access Denied", history

    message_lower = message.lower()

    # Tool Invocation Logic
    if "appointment" in message_lower or "schedule" in message_lower:
        response = get_appointment(patient_id)

    elif "test" in message_lower or "result" in message_lower:
        response = get_test_results(patient_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about appointments or test results."

    else:
        response = "🤖 I can help with appointments or test results."

    # Store history (same format as your code)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history,history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Hospital MCP Agent (Gradio Version)")

    patient_id = gr.Textbox(label="Enter Patient ID (e.g., P101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[msg, chatbot,state]
    )


# ================================
# STEP 7: Launch App
# ================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


SCENARIO: “Banking Smart Assistant System”
🏦 Background Story
A major bank launches an AI-powered customer assistant.
👉 Customers can ask:
- “What is my account balance?”
- “Show me my last 5 transactions.”
- “When is my loan EMI due?”
👉 Instead of logging into apps or waiting on customer service calls,
👉 AI fetches the information instantly, securely, and in real time.

⚙️ Core Idea:
Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
💡 Impact:
- Saves customers time.
- Reduces load on call centers.
- Provides personalized financial insights on demand.

In [13]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio


# ======================================
# STEP 2: Load API Key from env file
# ======================================
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found in env files")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history=history+ [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return "", history,history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot,state]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
